# 🏆 Modern Xiangqi Engine Championship Generator

This Google Colab notebook pits **Pikafish NNUE** (with its official 50MB neural network) against **Fairy-Stockfish Largeboard** across **60 classical Xiangqi opening systems** (120 games total) at high depth (**Depth 18 / NNUE**).

### How to Run:
1. Click **Runtime** -> **Run all** (or press `Ctrl+F9`).
2. The cloud VM will automatically download the engines, play the 120 grandmaster-level games, merge them with 27 verified online engine clashes (total 147 games), and automatically download `engine-games.json` to your browser!
3. Once downloaded, place `engine-games.json` into your XiangQi app assets folder.

In [ ]:
#@title Modern Xiangqi Engine Grand Championship Generator
#@markdown Runs official **Pikafish NNUE** vs **Fairy-Stockfish Largeboard** across 60 classical openings (120 games total) at higher depth on Google Cloud.
#@markdown Merges results with 27 verified online engine clashes to produce 147 games and automatically downloads `engine-games.json`.

import os
import sys
import time
import json
import subprocess
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

# --- Configuration ---
# Set the calculation depth and time controls
TARGET_DEPTH = 18       # Depth 16-20 ensures grandmaster / NNUE master calculation
MOVETIME_MS = 400       # Cap thinking time per move (ms) so games run efficiently
MAX_PLIES = 160         # 160 plies (80 full moves) adjudication limit
RESIGN_THRESHOLD = 1800 # Centipawn eval difference for resignation

print(f"=== Xiangqi Engine Championship Runner ===")
print(f"Engine 1: Pikafish NNUE (50MB official neural network)")
print(f"Engine 2: Fairy-Stockfish 14 LB (Largeboard Xiangqi)")
print(f"Target Depth: {TARGET_DEPTH} | Max Move Time: {MOVETIME_MS}ms")

OPENING_SEEDS = [
    # Central Cannon vs Screen Horse (1-8)
    ("Central Cannon vs Screen Horse: Pawn 7", ["b2e2", "b9c7", "c3c4", "h9g7"]),
    ("Central Cannon vs Screen Horse: Pawn 3", ["b2e2", "b9c7", "g3g4", "g6g5"]),
    ("Central Cannon vs Screen Horse: Double River Pawns", ["b2e2", "b9c7", "c3c4", "c6c5"]),
    ("Central Cannon vs Screen Horse: Edge Horse", ["b2e2", "b9c7", "b0c2", "h9i7"]),
    ("Central Cannon vs Screen Horse: Left Horse", ["b2e2", "b9c7", "b0c2", "h9g7"]),
    ("Central Cannon vs Screen Horse: Right Horse", ["b2e2", "b9c7", "h0g2", "h9g7"]),
    ("Central Cannon vs Screen Horse: Left Chariot Patrol", ["b2e2", "b9c7", "b0c2", "a9b9"]),
    ("Central Cannon vs Screen Horse: Patrol Cannon", ["b2e2", "b9c7", "b0c2", "b7b3"]),

    # Same Direction Cannons (9-14)
    ("Same Direction Cannons: Left Chariot", ["b2e2", "b7e7", "b0c2", "b9c7"]),
    ("Same Direction Cannons: Crossed Chariots", ["b2e2", "b7e7", "b0c2", "h9g7"]),
    ("Same Direction Cannons: Pawn 7 Attack", ["b2e2", "b7e7", "c3c4", "c6c5"]),
    ("Same Direction Cannons: Pawn 3 Attack", ["b2e2", "b7e7", "g3g4", "g6g5"]),
    ("Same Direction Cannons: Right Horse", ["b2e2", "b7e7", "h0g2", "h9g7"]),
    ("Same Direction Cannons: Fast Chariot", ["b2e2", "b7e7", "a0a1", "b9c7"]),

    # Opposite Direction Cannons (15-20)
    ("Opposite Direction Cannons: Standard Main Line", ["b2e2", "h7e7", "b0c2", "b9c7"]),
    ("Opposite Direction Cannons: Pawn 3 Strike", ["b2e2", "h7e7", "g3g4", "b9c7"]),
    ("Opposite Direction Cannons: Pawn 7 Strike", ["b2e2", "h7e7", "c3c4", "h9g7"]),
    ("Opposite Direction Cannons: Mutual Horse", ["b2e2", "h7e7", "b0c2", "h9g7"]),
    ("Opposite Direction Cannons: Edge Horse", ["b2e2", "h7e7", "b0c2", "b9a7"]),
    ("Opposite Direction Cannons: Right Horse", ["b2e2", "h7e7", "h0g2", "b9c7"]),

    # Sandwiched Horse & Single Horse (21-28)
    ("Central Cannon vs Sandwiched Horse: Main Line", ["b2e2", "h9g7", "b0c2", "b9c7"]),
    ("Central Cannon vs Sandwiched Horse: Pawn 7", ["b2e2", "h9g7", "c3c4", "b9c7"]),
    ("Central Cannon vs Sandwiched Horse: Pawn 3", ["b2e2", "h9g7", "g3g4", "g6g5"]),
    ("Central Cannon vs Sandwiched Horse: Left Chariot", ["b2e2", "h9g7", "b0c2", "a9a8"]),
    ("Central Cannon vs Single Horse: Left Edge Horse", ["b2e2", "b9a7", "b0c2", "h9g7"]),
    ("Central Cannon vs Single Horse: Pawn 7", ["b2e2", "b9a7", "c3c4", "h9g7"]),
    ("Central Cannon vs Single Horse: Pawn 3", ["b2e2", "b9a7", "g3g4", "c6c5"]),
    ("Central Cannon vs Single Horse: Right Edge Horse", ["b2e2", "h9i7", "b0c2", "b9c7"]),

    # Asymmetric Defenses (29-34)
    ("Central Cannon vs Three-Step Tiger", ["b2e2", "c9e7", "b0c2", "b9c7"]),
    ("Central Cannon vs Turtle-Back Cannon", ["b2e2", "b7b3", "b0c2", "b9c7"]),
    ("Central Cannon vs Left Elephant Setup", ["b2e2", "c9e7", "c3c4", "h9g7"]),
    ("Central Cannon vs Right Elephant Setup", ["b2e2", "g9e7", "b0c2", "b9c7"]),
    ("Central Cannon vs Corner Cannon Defense", ["b2e2", "b7d7", "b0c2", "b9c7"]),
    ("Central Cannon vs Patrol Cannon Defense", ["b2e2", "h7h4", "b0c2", "b9c7"]),

    # Pawn 7 / Minister Opening (35-42)
    ("Pawn 7 Opening vs Pawn 7 Defense", ["c3c4", "c6c5", "b0c2", "b9c7"]),
    ("Pawn 7 Opening vs Central Cannon", ["c3c4", "b7e7", "b0c2", "b9c7"]),
    ("Pawn 7 Opening vs Opposite Cannon", ["c3c4", "h7e7", "b0c2", "b9c7"]),
    ("Pawn 7 Opening vs Screen Horse", ["c3c4", "b9c7", "b0c2", "h9g7"]),
    ("Pawn 7 Opening vs Sandwiched Horse", ["c3c4", "h9g7", "b0c2", "b9c7"]),
    ("Pawn 7 Opening vs Golden Hook Cannon", ["c3c4", "b7a7", "b0c2", "b9c7"]),
    ("Pawn 7 Opening vs Edge Horse", ["c3c4", "b9a7", "b0c2", "h9g7"]),
    ("Pawn 7 Opening vs Left Elephant", ["c3c4", "c9e7", "b0c2", "b9c7"]),

    # Pawn 3 Opening (43-46)
    ("Pawn 3 Opening vs Pawn 3 Defense", ["g3g4", "g6g5", "h0g2", "h9g7"]),
    ("Pawn 3 Opening vs Central Cannon", ["g3g4", "b7e7", "h0g2", "b9c7"]),
    ("Pawn 3 Opening vs Screen Horse", ["g3g4", "b9c7", "b0c2", "h9g7"]),
    ("Pawn 3 Opening vs Elephant Defense", ["g3g4", "g9e7", "h0g2", "h9g7"]),

    # Elephant / Flying Bishop (47-52)
    ("Right Elephant vs Left Central Cannon", ["c0e2", "b7e7", "b0c2", "b9c7"]),
    ("Right Elephant vs Right Central Cannon", ["c0e2", "h7e7", "b0c2", "b9c7"]),
    ("Right Elephant vs Screen Horse", ["c0e2", "b9c7", "b0c2", "h9g7"]),
    ("Right Elephant vs Pawn 7 Push", ["c0e2", "c6c5", "c3c4", "b9c7"]),
    ("Left Elephant vs Central Cannon", ["g0e2", "b7e7", "h0g2", "b9c7"]),
    ("Left Elephant vs Screen Horse", ["g0e2", "b9c7", "b0c2", "h9g7"]),

    # Horse & Flank Openings (53-60)
    ("Right Horse Opening vs Central Cannon", ["b0c2", "b7e7", "h2e2", "b9c7"]),
    ("Right Horse Opening vs Pawn 7", ["b0c2", "c6c5", "c3c4", "b9c7"]),
    ("Right Horse Opening vs Screen Horse", ["b0c2", "b9c7", "c3c4", "h9g7"]),
    ("Left Horse Opening vs Central Cannon", ["h0g2", "b7e7", "b2e2", "b9c7"]),
    ("Cross-Palace Cannon vs Central Cannon", ["b2d2", "b7e7", "b0c2", "b9c7"]),
    ("Cross-Palace Cannon vs Screen Horse", ["b2d2", "b9c7", "b0c2", "h9g7"]),
    ("Scholar-Corner Cannon vs Central Cannon", ["b2f2", "b7e7", "b0c2", "b9c7"]),
    ("Scholar-Corner Cannon vs Screen Horse", ["b2f2", "b9c7", "b0c2", "h9g7"])
]

ONLINE_CANDIDATES = [
  {
    "id": "online-engine-001",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 1",
    "round": "Game 1",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "g3g4",
      "h7g7",
      "b2e2",
      "c9e7",
      "h0i2",
      "h9i7",
      "i0h0",
      "i9h9",
      "h2h6",
      "d9e8",
      "b0c2",
      "c6c5",
      "e2e6",
      "b9c7",
      "e6e5",
      "b7b5",
      "e5b5",
      "c7b5",
      "a0b0",
      "b5c7",
      "h6h7",
      "a9d9",
      "b0b6",
      "d9d2",
      "b6g6",
      "g7f7",
      "c2b0",
      "d2d1",
      "i2g3",
      "d1b1",
      "g3e4",
      "b1b0",
      "g6c6",
      "c7a8",
      "c6a6",
      "a8c7",
      "a6c6",
      "c7a8",
      "g0e2",
      "b0b6",
      "c6f6",
      "b6f6",
      "e4f6",
      "f7f8",
      "g4g5",
      "e8f7",
      "f6d7",
      "f8d8",
      "g5g6",
      "a8c7",
      "g6h6",
      "i6i5",
      "e3e4",
      "h9h8",
      "h6i6",
      "f7e8",
      "d7e5",
      "c7e6",
      "h0h6",
      "e6g5",
      "i6i7",
      "g5e4",
      "d0e1",
      "h8h9",
      "h6d6",
      "g9i7",
      "h7h6",
      "e4g5",
      "e2g4",
      "d8c8",
      "d6c6",
      "c8d8",
      "a3a4",
      "g5f7",
      "e5f7",
      "e8f7",
      "a4a5",
      "f9e8",
      "a5b5",
      "h9g9",
      "b5c5",
      "g9g4",
      "c5d5",
      "g4b4",
      "h6e6",
      "b4b0",
      "e1d0",
      "b0c0",
      "f0e1",
      "c0c2",
      "d5d6",
      "e9f9",
      "c6c8",
      "d8d9",
      "e0f0",
      "e7g5",
      "e6e4",
      "i7g9",
      "e4f4",
      "f9e9",
      "f4b4",
      "c2b2",
      "b4e4",
      "g9e7",
      "d6d7",
      "b2e2",
      "c8c4",
      "e2i2",
      "d7d8",
      "i2i0",
      "f0f1",
      "i0i3",
      "e4e6",
      "i3f3",
      "e1f2",
      "f3d3",
      "c4c8",
      "e9f9",
      "e6f6",
      "f9e9",
      "f6a6",
      "e9f9",
      "a6a9",
      "f9f8",
      "a9a4",
      "f8f9",
      "d0e1",
      "d3d6",
      "a4a9",
      "f9f8",
      "a9a5",
      "f8f9",
      "c3c4",
      "i5i4",
      "c4c5",
      "d6a6",
      "a5b5",
      "a6b6",
      "d8d9",
      "e8d9",
      "b5a5",
      "b6a6",
      "c5b5",
      "d9e8",
      "f1f0",
      "i4i3",
      "c8c3",
      "i3i2",
      "c3h3",
      "f9e9",
      "f0e0",
      "a6c6",
      "a5a9",
      "c6c0",
      "e1d0",
      "c0b0",
      "b5a5",
      "b0b8",
      "a9a7",
      "b8b6",
      "f2e1",
      "b6b7",
      "a7a6",
      "b7b6",
      "a6a9",
      "b6b8",
      "h3h9",
      "e8f9",
      "a9f9",
      "e7g9",
      "h9g9",
      "b8h8",
      "e0f0",
      "i2h2",
      "g9g5",
      "e9f9",
      "g5g7",
      "h2g2",
      "g7g2",
      "h8h0",
      "f0f1",
      "h0h5",
      "a5a6",
      "h5h6",
      "g2g9",
      "f9f8",
      "g9g8",
      "f8f9",
      "a6a7",
      "h6f6",
      "e1f2",
      "f6e6",
      "g8g9",
      "f9f8",
      "g9g7",
      "f7e8",
      "a7b7",
      "e6e5",
      "g7i7",
      "e5e4",
      "b7c7",
      "e4e2",
      "d0e1",
      "e2e6",
      "f1f0",
      "e6e4",
      "i7i8",
      "f8f9",
      "i8i9",
      "f9f8",
      "i9i7",
      "e4e5",
      "i7g7",
      "e5e4",
      "f0e0",
      "e4e5",
      "e0d0",
      "e5e4",
      "g7g8",
      "f8f9",
      "g8g9",
      "f9f8",
      "g9g7",
      "e4d4",
      "d0e0",
      "d4e4"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1883846.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-002",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 2",
    "round": "Game 2",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "c3c4",
      "b7c7",
      "g0e2",
      "b9a7",
      "b0c2",
      "h7e7",
      "a0b0",
      "h9g7",
      "h2f2",
      "i9h9",
      "b2a2",
      "h9h1",
      "f0e1",
      "a9b9",
      "b0b9",
      "a7b9",
      "a2a1",
      "h1h5",
      "h0g2",
      "c6c5",
      "c4c5",
      "h5c5",
      "a1c1",
      "c5h5",
      "f2f1",
      "d9e8",
      "g3g4",
      "g6g5",
      "g4g5",
      "h5g5",
      "i0g0",
      "c9a7",
      "g2i1",
      "g5g0",
      "i1g0",
      "a7c5",
      "c2b4",
      "c7c1",
      "f1c1",
      "e7e3",
      "g0h2",
      "e3h3",
      "b4a6",
      "c5e7",
      "a6b4",
      "g7h5",
      "i3i4",
      "b9d8",
      "e1d2",
      "h5f4",
      "a3a4",
      "d8b7",
      "h2f3",
      "b7c5",
      "c1f1",
      "f4h5",
      "f3h4",
      "h3e3",
      "f1e1",
      "e3e1",
      "d0e1",
      "c5a4",
      "b4d3",
      "g9i7",
      "h4f5",
      "e7g9",
      "e2g0",
      "a4c3",
      "d3c5",
      "c3d5",
      "f5h4",
      "e6e5",
      "c5d3",
      "e5e4",
      "d3e5",
      "e4f4",
      "c0e2",
      "d5e7",
      "e0d0",
      "e9d9",
      "d0e0",
      "e7g8",
      "e5g6",
      "g8f6",
      "h4g2",
      "f4e4",
      "i4i5",
      "i6i5",
      "g6i5",
      "h5g7",
      "g2h4",
      "f6d5",
      "i5g6",
      "g7e6",
      "g6e5",
      "g9e7",
      "h4g6",
      "i7g9",
      "g6f8",
      "e6g5",
      "e2g4",
      "d9e9",
      "f8g6",
      "d5c7",
      "e5c6",
      "e8f7",
      "g6e5",
      "c7a6",
      "e5c4",
      "f9e8",
      "c4d6",
      "a6c5",
      "c6a5",
      "e4f4",
      "g4i2",
      "c5d3",
      "a5b3",
      "d3b4",
      "b3c1",
      "e8d7",
      "c1e2",
      "g5e6",
      "d6c4",
      "f4e4",
      "c4d6",
      "e4d4",
      "e0f0",
      "b4d5",
      "d6b5",
      "d4e4",
      "e2g3",
      "e4f4",
      "f0e0",
      "d5b4",
      "g3e2",
      "e7c5",
      "e1d0",
      "e6d4",
      "b5c7",
      "d4c6",
      "d0e1",
      "f4e4",
      "e2g3",
      "e4d4",
      "g3f5",
      "c5a7",
      "f5d6",
      "f7e8",
      "g0e2",
      "b4d5",
      "i2g0",
      "g9e7",
      "e0f0",
      "e8f9",
      "f0e0",
      "e9e8",
      "e0f0",
      "d4d3",
      "e2c0",
      "e7c9",
      "c0e2",
      "d3e3",
      "e2c0"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1774681.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-003",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 3",
    "round": "Game 3",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2e2",
      "h9g7",
      "h0g2",
      "i9h9",
      "i0h0",
      "g6g5",
      "h0h6",
      "b9c7",
      "c3c4",
      "h7i7",
      "h6g6",
      "i7i8",
      "b0c2",
      "d9e8",
      "b2a2",
      "a9b9",
      "a0b0",
      "i8g8",
      "g6f6",
      "g7h5",
      "e2e6",
      "c7e6",
      "f6e6",
      "g5g4",
      "g3g4",
      "h5f4",
      "g2f4",
      "g8g0",
      "f0e1",
      "b7b1",
      "e6f6",
      "g0i0",
      "f4g2",
      "b9b2",
      "f6f2",
      "h9h0",
      "g2f0",
      "h0h2",
      "f0g2",
      "g9e7",
      "a2a6",
      "i0i2",
      "c0e2",
      "i2g2",
      "e2g0",
      "g2c2",
      "f2h2",
      "c2c0",
      "b0c0",
      "b2h2",
      "c0b0",
      "b1c1",
      "e1f0",
      "h2h3",
      "b0b3",
      "c1i1",
      "i3i4",
      "e8d9",
      "a6a9",
      "f9e8",
      "a3a4",
      "i1i0",
      "a4a5",
      "h3g3",
      "b3b7",
      "g3e3",
      "d0e1",
      "e3e4",
      "b7b3",
      "e4c4",
      "b3i3",
      "i0h0",
      "i3h3",
      "h0i0",
      "h3h0",
      "i0i1",
      "h0h9",
      "e8f9",
      "h9h6",
      "c4g4",
      "h6e6",
      "g4g7",
      "a5b5",
      "i1i0",
      "e1d2",
      "f9e8",
      "e0e1",
      "g7g1",
      "e1e2",
      "g1g2",
      "e2e1",
      "g2g7",
      "a9b9",
      "c6c5",
      "b5c5",
      "g7g1",
      "e1e2",
      "g1b1",
      "b9b6",
      "e7c5",
      "b6i6",
      "i0f0",
      "e6h6",
      "c5e7",
      "h6h9",
      "f0f9",
      "i6i9",
      "b1g1",
      "h9h1",
      "g1g9",
      "i9f9",
      "g9f9",
      "h1h6",
      "e8f7",
      "e2e1",
      "f9i9",
      "h6h0",
      "c9a7",
      "h0i0",
      "i9g9",
      "g0e2",
      "g9g1",
      "e1e0",
      "e9e8",
      "i0i2",
      "g1b1",
      "e0d0",
      "b1b0",
      "d0d1",
      "a7c5",
      "e2g4",
      "e8d8",
      "i2e2",
      "b0b1",
      "d1d0",
      "b1b2",
      "d0d1",
      "b2b0",
      "d1e1",
      "e7g5",
      "e2i2",
      "b0b1",
      "e1e2",
      "d9e8",
      "i2i3",
      "e8d7",
      "i4i5",
      "b1g1",
      "i5h5",
      "g1g2",
      "e2e1",
      "g2g4",
      "i3i0",
      "g4e4",
      "e1d1",
      "e4b4",
      "i0i6",
      "b4b1",
      "d1d0",
      "b1b0",
      "d0d1",
      "g5e7",
      "i6i2",
      "b0b6",
      "h5h6",
      "b6b1",
      "d1d0",
      "b1b2",
      "d0d1",
      "d7e8",
      "i2h2",
      "c5a7",
      "d1d0",
      "e7c5",
      "h2i2",
      "e8d7",
      "i2h2",
      "c5e7",
      "h2i2",
      "b2b0",
      "d0d1",
      "b0b1",
      "d1d0",
      "b1b2",
      "i2h2",
      "f7e8",
      "h2i2",
      "b2b0",
      "d0d1",
      "b0b1",
      "d1d0",
      "a7c9",
      "i2h2",
      "b1b0",
      "d0d1",
      "b0b6",
      "h2i2",
      "b6h6",
      "i2e2",
      "d8d9",
      "e2e4",
      "h6b6",
      "e4d4",
      "b6b1",
      "d1d0",
      "d9e9",
      "d0e0",
      "b1b0",
      "e0e1",
      "b0b6",
      "d4e4",
      "b6b1",
      "e1e0",
      "b1b0",
      "e0e1",
      "b0b6",
      "e1e0",
      "e7g9",
      "e0d0",
      "b6b0",
      "d0d1",
      "b0b2",
      "d2e1",
      "b2b1",
      "d1d2",
      "b1b6",
      "d2d1",
      "g9e7",
      "e1f2",
      "b6b1",
      "d1d0",
      "b1b2",
      "f2e1",
      "b2b0",
      "d0d1",
      "b0b1",
      "d1d2",
      "b1b6",
      "d2d1",
      "e7g9",
      "e4e2",
      "b6b0",
      "e1f2",
      "g9e7",
      "d1e1",
      "b0b8",
      "e1e0",
      "b8b3",
      "e0e1",
      "b3b0",
      "e1f1",
      "b0b6",
      "e2e4",
      "e7g9",
      "e4e2",
      "e9f9",
      "e2c2",
      "g9e7",
      "c2e2",
      "b6g6",
      "e2e4",
      "e8d9",
      "f1f0",
      "g6b6",
      "e4e2",
      "b6b0",
      "f0f1",
      "b0b1",
      "f1f0",
      "b1g1",
      "f0e0",
      "g1g6",
      "e0e1",
      "f9e9",
      "e1f1",
      "g6b6",
      "e2e4",
      "d9e8",
      "e4e5",
      "b6b1",
      "f1f0",
      "b1b0",
      "f0f1",
      "b0g0",
      "e5e6",
      "g0g1",
      "f1f0",
      "g1i1",
      "f0e0",
      "i1i2",
      "e6e2",
      "i2i0",
      "e0e1",
      "i0i1",
      "e1e0",
      "i1g1",
      "e2d2",
      "g1b1",
      "d2e2",
      "e9f9",
      "e0d0",
      "e8f7",
      "d0e0",
      "b1b0",
      "e0e1",
      "d7e8",
      "e2e6",
      "b0b1",
      "e1e2",
      "b1b2",
      "e2e1",
      "b2f2",
      "e6e2",
      "f2e2",
      "e1e2"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1917417.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-004",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 4",
    "round": "Game 4",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "g3g4",
      "c6c5",
      "h0g2",
      "b9c7",
      "c0e2",
      "h9g7",
      "g2f4",
      "a9a8",
      "b0d1",
      "i9i8",
      "i0i1",
      "a8d8",
      "f4g6",
      "d8d4",
      "b2c2",
      "i8f8",
      "i1g1",
      "f8f3",
      "a0b0",
      "c7e8",
      "c2c0",
      "b7d7",
      "h2h4",
      "d4d5",
      "d1b2",
      "e8d6",
      "d0e1",
      "d7b7",
      "b2a4",
      "c9a7",
      "b0b6",
      "d6c8",
      "b6b5",
      "a6a5",
      "b5a5",
      "b7b0",
      "c0c2",
      "d5d1",
      "a5b5",
      "b0a0",
      "c2d2",
      "h7h8",
      "b5b9",
      "h8e8",
      "g1g2",
      "e8e3",
      "g2f2",
      "f3h3",
      "f2h2",
      "h3g3",
      "h2g2",
      "g3i3",
      "g2i2",
      "i3f3",
      "i2f2",
      "f3g3",
      "f2g2",
      "g3h3",
      "g2h2",
      "h3h2",
      "d2h2",
      "d1c1",
      "e0d0",
      "f9e8",
      "b9b7",
      "e3a3",
      "h2h1",
      "c1c0",
      "d0d1",
      "a3a1",
      "b7g7",
      "a1b1",
      "g7g9",
      "e8f9",
      "d1d2",
      "c0c2",
      "d2d1",
      "c2c1",
      "d1d2",
      "c1c2",
      "d2d1",
      "c2c0",
      "d1d2",
      "c0c2",
      "d2d1",
      "c2c0",
      "d1d2",
      "c0c2",
      "d2d1",
      "c2c0",
      "d1d2",
      "c0c2",
      "d2d1",
      "c2c1",
      "d1d2",
      "c1c3",
      "a4c3",
      "a0g0",
      "e2g0",
      "b1h1",
      "g9i9",
      "h1i1",
      "g0i2",
      "c5c4",
      "h4h9",
      "f9e8",
      "h9h3",
      "e8f9",
      "g6h8",
      "d9e8",
      "h3e3",
      "c8e7",
      "e3e7",
      "e8f7",
      "i9i8",
      "e9e8",
      "h8g6",
      "e8e9",
      "i8d8",
      "c4c3",
      "g6h8",
      "c3c2",
      "d2e2",
      "c2d2",
      "e1d2",
      "i6i5",
      "h8f7"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908579.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-005",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 5",
    "round": "Game 5",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h0g2",
      "g6g5",
      "b2e2",
      "b9c7",
      "b0c2",
      "a9b9",
      "a0b0",
      "b7b3",
      "c3c4",
      "h9g7",
      "c2d4",
      "i9i8",
      "c4c5",
      "b3b2",
      "h2h6",
      "c6c5",
      "h6g6",
      "g7e8",
      "i0h0",
      "h7e7",
      "h0h6",
      "i8f8",
      "g6i6",
      "c5c4",
      "d4e6",
      "f8f5",
      "g2e1",
      "c4c3",
      "e6c7",
      "e8c7",
      "h6c6",
      "b9b7",
      "e2e7",
      "c9e7",
      "c6c3",
      "c7d5",
      "c3d3",
      "b2e2",
      "e1c2",
      "b7b0",
      "c2b0",
      "d5f4",
      "d3d1",
      "e2b2",
      "d1f1",
      "f4d3",
      "f1d1",
      "d3c5",
      "i3i4",
      "b2b7",
      "i6i5",
      "f5f4",
      "i5c5",
      "e7c5",
      "d1d7",
      "b7a7",
      "d7d5",
      "g9e7",
      "i4i5",
      "f4b4",
      "b0c2",
      "b4b3",
      "i5h5",
      "b3c3",
      "c2e1",
      "a7a3",
      "e3e4",
      "d9e8",
      "h5g5",
      "c3g3",
      "d5d3",
      "g3d3",
      "e1d3",
      "e7g5",
      "d3c5",
      "a6a5",
      "e4e5",
      "a3h3",
      "c5b3",
      "a5a4",
      "f0e1",
      "h3h0",
      "g0e2",
      "h0h5",
      "e5e6",
      "e9d9",
      "e6f6",
      "d9e9",
      "b3d4",
      "a4b4",
      "e0f0",
      "b4a4",
      "e2g4",
      "a4b4",
      "c0e2",
      "e9d9",
      "e1d2",
      "h5h7",
      "d0e1",
      "g5i7",
      "d4e6",
      "i7g5",
      "e6c7",
      "d9d8",
      "f6g6",
      "e8d7",
      "c7a6",
      "b4b3",
      "a6c5",
      "b3b2",
      "c5e6",
      "g5e7",
      "e6f4",
      "f9e8",
      "f4d3",
      "b2b1",
      "d3e5",
      "d8d9",
      "g6f6",
      "d9e9",
      "f6e6",
      "e7c9",
      "e6d6",
      "b1c1",
      "e2g0",
      "c1d1",
      "g0i2",
      "e9d9",
      "g4e2",
      "d9d8",
      "d6e6",
      "h7h9",
      "i2g4",
      "d8d9",
      "e6d6",
      "d1c1",
      "e5g6",
      "d9e9",
      "e2c4",
      "h9f9",
      "f0e0",
      "c1d1",
      "g6h8",
      "f9h9",
      "e1f2",
      "e9d9",
      "e0f0",
      "h9e9",
      "g4i2",
      "e9g9",
      "f0e0",
      "g9g8",
      "c4e2",
      "d9d8",
      "d6e6",
      "g8g9",
      "e6f6",
      "g9g7",
      "f2e1",
      "g7h7",
      "i2g0",
      "d8d9",
      "f6e6",
      "h7h0",
      "g0i2",
      "h0h6",
      "e6d6",
      "h6h7",
      "e0f0",
      "c9a7",
      "i2g4",
      "d9e9",
      "e1f2",
      "a7c9",
      "e2g0",
      "e9d9",
      "f2e1",
      "h7h0",
      "g0e2",
      "d9e9",
      "h8i6",
      "h0h9",
      "i6h8",
      "h9g9",
      "f0e0",
      "g9g8",
      "h8g6",
      "e9d9",
      "g6e5",
      "g8f8",
      "g4i2",
      "f8h8",
      "e0f0",
      "h8g8",
      "e5g6",
      "d9d8",
      "e2c4",
      "g8g9",
      "g6e5",
      "g9e9"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908558.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-006",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 6",
    "round": "Game 6",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "c0e2",
      "h7f7",
      "g3g4",
      "h9i7",
      "h0g2",
      "i9h9",
      "i0h0",
      "h9h5",
      "h2i2",
      "h5d5",
      "c3c4",
      "c9e7",
      "d0e1",
      "c6c5",
      "c4c5",
      "d5c5",
      "e3e4",
      "b9c7",
      "g2e3",
      "c7b5",
      "b2b7",
      "f7b7",
      "h0h3",
      "b5c3",
      "e3c4",
      "c3e4",
      "i2i6",
      "d9e8",
      "h3e3",
      "c5i5",
      "i6e6",
      "e4c5",
      "i3i4",
      "i5i4",
      "b0c2",
      "g6g5",
      "g4g5",
      "i4i5",
      "e6e5",
      "i5g5",
      "e3f3",
      "i7g6",
      "f3f6",
      "g6i5",
      "a0b0",
      "g5g6",
      "f6f5",
      "i5h3",
      "b0b3",
      "h3g1",
      "f5f1",
      "b7c7",
      "c2e3",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1",
      "f3f1",
      "g1h3",
      "f1f3",
      "h3g1"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908549.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-007",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 7",
    "round": "Game 7",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2f2",
      "g6g5",
      "h0g2",
      "h9g7",
      "i0h0",
      "i9h9",
      "c3c4",
      "h7h3",
      "b0c2",
      "b7e7",
      "b2a2",
      "b9a7",
      "a0b0",
      "a9b9",
      "b0b9",
      "a7b9",
      "a2a6",
      "b9c7",
      "a6a4",
      "h3e3",
      "h0h9",
      "e3c3",
      "d0e1",
      "g7h9",
      "c0e2",
      "e6e5",
      "g3g4",
      "g5g4",
      "e2g4",
      "c6c5",
      "c4c5",
      "c7a6",
      "a4a5",
      "e5e4",
      "c5d5",
      "e7a7",
      "a5a7",
      "c9a7",
      "g4e2",
      "a6b4",
      "d5d6",
      "b4d3",
      "f2f1",
      "h9g7",
      "g2h4",
      "c3i3",
      "h4g6",
      "f9e8",
      "f1g1",
      "g7i8",
      "a3a4",
      "a7c9",
      "a4a5",
      "g9e7",
      "g1i1",
      "i8g7",
      "a5b5",
      "e7g5",
      "i1g1",
      "d3f4",
      "g1g4",
      "i3i4",
      "b5c5",
      "g7i8",
      "g6f8",
      "g5e7",
      "c5c6",
      "i8h6",
      "g4h4",
      "h6f5",
      "h4h9",
      "f4h3",
      "e0d0",
      "h3g1",
      "f8g6",
      "g1f3",
      "d0d1",
      "i4i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "i1i0",
      "f2e1",
      "h2f3",
      "d0d1",
      "i0i1",
      "e1f2",
      "f3h2",
      "d1d0",
      "h2i0",
      "c2b4",
      "i0g1",
      "f0e1",
      "g1i0",
      "b4d5",
      "i0g1",
      "d5f6",
      "g1i0",
      "g6h8"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908546.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-008",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 8",
    "round": "Game 8",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "b0a2",
      "h7e7",
      "h0g2",
      "b9c7",
      "g3g4",
      "h9g7",
      "i0h0",
      "i9h9",
      "h2h6",
      "g7i8",
      "h6h5",
      "c6c5",
      "d0e1",
      "h9h6",
      "b2c2",
      "a9b9",
      "a0b0",
      "b7b5",
      "h5b5",
      "e7e3",
      "c0e2",
      "h6h0",
      "g2h0",
      "b9b5",
      "c3c4",
      "c9e7",
      "h0g2",
      "e3e4",
      "b0b5",
      "c7b5",
      "c4c5",
      "e7c5",
      "c2b2",
      "c5e7",
      "b2b4",
      "e4e5",
      "a2c3",
      "b5d4",
      "c3d5",
      "i8g7",
      "g2e3",
      "d9e8",
      "e3c4",
      "d4f3",
      "d5b6",
      "e5h5",
      "b4b1",
      "h5h8",
      "c4d6",
      "g6g5",
      "g4g5",
      "f3g5",
      "b1d1",
      "e8d7",
      "b6d7",
      "e9e8",
      "d7b6",
      "e8f8",
      "d6c8",
      "e7c9",
      "b6d7",
      "c9e7",
      "d7b6",
      "e7c9",
      "b6d7",
      "c9e7",
      "d7b6",
      "e7c9",
      "b6d7",
      "c9e7",
      "d7b6",
      "e7c9",
      "b6d7",
      "c9e7",
      "d7b6",
      "g7f5",
      "e1f2",
      "f8e8",
      "d1a1",
      "h8h6",
      "b6d5",
      "h6h8",
      "d5f6",
      "g5f7",
      "c8b6",
      "f5d4",
      "a1i1",
      "h8i8",
      "i1i6",
      "i8i3",
      "b6d5",
      "e8e9",
      "i6e6",
      "f9e8",
      "a3a4",
      "i3d3",
      "e6d6",
      "d3d5",
      "f6d5",
      "d4b3",
      "d5b4",
      "b3d2",
      "e0e1",
      "d2e4",
      "d6d4",
      "f7g5",
      "b4a6",
      "g5e6",
      "d4c4",
      "e4c5",
      "a6c5",
      "e6c5",
      "a4a5",
      "c5b7",
      "a5b5",
      "b7d6",
      "b5a5",
      "d6c4",
      "a5b5",
      "e9d9",
      "e1f1",
      "c4d6",
      "b5a5",
      "d9e9",
      "e2c0",
      "e9f9",
      "f0e1",
      "d6e4",
      "c0a2",
      "e4c3",
      "a2c4",
      "c3e4",
      "f1f0",
      "e4d6",
      "c4a2",
      "d6b7",
      "a5a6",
      "b7c5",
      "a6b6",
      "c5d7",
      "b6b7",
      "d7c5",
      "b7b8",
      "c5d7",
      "b8a8",
      "g9i7",
      "a2c0",
      "i7g5",
      "f0e0",
      "e7c5",
      "e1f0",
      "e8f7",
      "e0d0",
      "f9e9",
      "f0e1",
      "c5a7",
      "e1d2",
      "g5e7",
      "f2e1",
      "e7c5",
      "d0e0",
      "e9e8",
      "e0f0",
      "d7e5",
      "g0i2",
      "e5f3",
      "a8b8",
      "f3g1",
      "i2g0",
      "g1h3",
      "e1d0",
      "h3f4",
      "f0f1",
      "f4g2",
      "d0e1",
      "g2e3",
      "c0e2",
      "e3f5",
      "f1f0",
      "f5d6",
      "e1d0",
      "e8e7",
      "f0e0",
      "f7e8",
      "d2e1",
      "d6f5",
      "e2c0",
      "f5d4",
      "e0f0",
      "d4f3",
      "e1f2",
      "f3d2",
      "c0e2",
      "d2e4",
      "d0e1",
      "e7f7",
      "e2c0",
      "e4g3",
      "f0e0",
      "g3h5",
      "e1f0",
      "h5f4",
      "f2e1",
      "f4d3",
      "g0i2",
      "d3c1",
      "e0d0",
      "c1b3",
      "d0d1",
      "b3d4",
      "e1f2",
      "d4c2",
      "f0e1",
      "c2e3",
      "d1d0",
      "e3c2",
      "d0d1",
      "c2e3",
      "d1d0",
      "e3c2",
      "d0d1",
      "c2e3",
      "d1d0",
      "e3c2",
      "d0d1",
      "c2d4",
      "e1d0",
      "d4c2",
      "d1d2",
      "c2b4",
      "b8c8",
      "b4d5",
      "d2d1",
      "c5e7",
      "d0e1",
      "d5f6",
      "e1f0",
      "f6g4",
      "f2e1",
      "g4h2",
      "e1d2"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908545.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-009",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 9",
    "round": "Game 9",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2i2",
      "b7d7",
      "h0g2",
      "h9g7",
      "i0h0",
      "i9h9",
      "g3g4",
      "b9c7",
      "b0a2",
      "a9b9",
      "b2c2",
      "b9b5",
      "a0b0",
      "b5f5",
      "c2c6",
      "c9e7",
      "a3a4",
      "g6g5",
      "g4g5",
      "f5g5",
      "g2f4",
      "g5f5",
      "b0b4",
      "h7h2",
      "g0e2",
      "d9e8",
      "f0e1",
      "e6e5",
      "f4d5",
      "f5f6",
      "d5c7",
      "f6c6",
      "b4b9",
      "d7d9",
      "c7d9",
      "e8d9",
      "a2b4",
      "c6c3",
      "b9b8",
      "g7e6",
      "b8b6",
      "e6g5",
      "i2i6",
      "g5i6",
      "b6i6",
      "c3e3",
      "i6a6",
      "e3i3",
      "b4d5",
      "i3d3",
      "d5b6",
      "e5e4",
      "b6c4",
      "d3d5",
      "a6e6",
      "h9h4",
      "c4b6",
      "d5d8",
      "a4a5",
      "e4f4",
      "e6f6",
      "d9e8",
      "a5a6",
      "h2h3",
      "b6c4",
      "d8d3",
      "a6b6",
      "e9d9",
      "f6d6",
      "d3d6",
      "c4d6",
      "h3h1",
      "b6c6",
      "f4f3",
      "c0a2",
      "h4h5",
      "d6e4",
      "e8f7",
      "c6c7",
      "f9e8",
      "c7c8",
      "h5h2",
      "a2c0",
      "h2h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6",
      "h4h5",
      "d6e4",
      "h5h4",
      "e4d6"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1908544.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-010",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 10",
    "round": "Game 10",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h0g2",
      "g6g5",
      "c3c4",
      "h9g7",
      "b0c2",
      "i9i8",
      "i0i1",
      "i8c8",
      "h2h6",
      "c6c5",
      "h6g6",
      "c5c4",
      "g6g9",
      "f9e8",
      "i1h1",
      "h7h5",
      "i3i4",
      "c9e7",
      "g9i9",
      "e8f7",
      "g2i3",
      "c8i8",
      "i4i5",
      "h5h8",
      "i9i6",
      "g7i6",
      "i5i6",
      "h8f8",
      "h1h9",
      "f8f9",
      "i3h5",
      "b9c7",
      "c2e1",
      "c7d5",
      "b2i2",
      "i8f8",
      "h5i7",
      "b7b3",
      "e3e4",
      "f7e8",
      "i6h6",
      "a9a7",
      "a0a1",
      "d5c3",
      "i7g6",
      "f8f6",
      "i2i9",
      "b3g3",
      "e1d3",
      "g3g6",
      "h6g6",
      "f6g6",
      "d3f4",
      "g6g9",
      "a1h1",
      "c3e4",
      "f0e1",
      "f9h9",
      "h1h9",
      "e8f9",
      "i9g9",
      "e7g9",
      "h9h6",
      "c4c3",
      "f4e6",
      "g5g4",
      "c0e2",
      "g4g3",
      "h6h5",
      "e4f6",
      "h5c5",
      "a7e7",
      "e6d4",
      "e7e3",
      "d4b5",
      "c3b3",
      "c5c6",
      "f6d7",
      "b5a7",
      "d9e8",
      "a7c8",
      "e9d9",
      "c6a6",
      "g9e7",
      "a3a4",
      "e7c9",
      "a6d6",
      "g3g2",
      "c8b6",
      "e3e7",
      "b6d7",
      "e7d7",
      "d6d7",
      "e8d7",
      "a4a5",
      "b3c3",
      "a5b5",
      "c3d3",
      "b5c5",
      "d3e3",
      "c5d5",
      "e3f3",
      "d5e5",
      "d7e8",
      "e5f5",
      "g2g1",
      "e0f0",
      "d9e9",
      "f5g5",
      "f3g3",
      "g5h5",
      "c9e7",
      "e1d2",
      "e9d9",
      "h5i5",
      "g3f3",
      "i5h5",
      "e8f7",
      "h5h6",
      "d9e9",
      "h6g6",
      "e7g5",
      "g6h6",
      "f3e3",
      "h6g6",
      "e3d3",
      "d2e1",
      "f7e8",
      "g0i2",
      "e8d7",
      "i2g0",
      "g1h1",
      "g6f6",
      "e9d9",
      "f6g6",
      "g5e7",
      "g6h6",
      "d7e8",
      "h6i6",
      "d9d8",
      "i6h6",
      "d3e3",
      "h6i6",
      "h1g1",
      "i6h6",
      "e8d9",
      "h6i6",
      "d8e8",
      "i6h6",
      "e7c5",
      "h6i6",
      "e3d3",
      "i6h6",
      "e8e9",
      "h6i6",
      "c5e7",
      "i6h6",
      "f9e8",
      "h6i6",
      "d3c3",
      "i6h6",
      "e8d7",
      "h6i6",
      "c3b3",
      "i6h6",
      "g1h1",
      "h6i6",
      "b3b2",
      "i6h6",
      "b2b1",
      "f0f1",
      "d9e8",
      "f1f2"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1893460.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-011",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 11",
    "round": "Game 11",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2e2",
      "h9g7",
      "c3c4",
      "g6g5",
      "b2d2",
      "i9h9",
      "b0c2",
      "b9c7",
      "a0b0",
      "a9b9",
      "h0g2",
      "b7b1",
      "i0h0",
      "h7h3",
      "e2e1",
      "b1b3",
      "e1g1",
      "h9h5",
      "g3g4",
      "g5g4",
      "g1g4",
      "h3g3",
      "h0h5",
      "g7h5",
      "g0e2",
      "c9e7",
      "f0e1",
      "b3c3",
      "b0b9",
      "c7b9",
      "d2d6",
      "b9c7",
      "d6i6",
      "c6c5",
      "c4c5",
      "e7c5",
      "i3i4",
      "h5g7",
      "i4i5",
      "g7i6",
      "i5i6",
      "c7b5",
      "g2i3",
      "c5e7",
      "i3h5",
      "b5d4",
      "c2b0",
      "d4f5",
      "h5g3",
      "c3g3",
      "b0c2",
      "a6a5",
      "g4b4",
      "g3g8",
      "b4b8",
      "g8d8",
      "b8a8",
      "d9e8",
      "a8a6",
      "d8b8",
      "i6h6",
      "b8b3",
      "h6g6",
      "b3c3",
      "c2a1",
      "f5e3",
      "a1b3",
      "c3c7",
      "b3a5",
      "c7a7",
      "a5c6",
      "a7a3",
      "a6e6",
      "e3d5",
      "c6b8",
      "a3a8",
      "e6e5",
      "a8a3",
      "e5e6",
      "a3a8",
      "e6e5",
      "a8a3",
      "g6f6",
      "a3c3",
      "b8a6",
      "c3e3",
      "c0a2",
      "e3i3",
      "e1d2",
      "i3i6",
      "a6b8",
      "i6i8",
      "b8a6",
      "i8i6",
      "a6b8",
      "d5c7",
      "d0e1",
      "c7d5",
      "a2c0",
      "d5b4",
      "b8d7",
      "e9d9",
      "d7b8",
      "d9e9",
      "e2c4",
      "b4c6",
      "b8d7",
      "e9d9",
      "e5d5",
      "c6d8",
      "d7e5",
      "d8c6",
      "e5g4",
      "d9e9",
      "c0e2",
      "c6d4",
      "g4e5",
      "i6i8",
      "d5a5",
      "i8i6",
      "e5d3",
      "i6i8",
      "d3f4",
      "e8d7",
      "f4d5",
      "e7c5",
      "a5a4",
      "d4f5",
      "a4a1",
      "f9e8",
      "f6e6",
      "i8i3",
      "a1c1",
      "c5e7",
      "e1f2",
      "i3d3",
      "c1i1",
      "e7c5",
      "i1e1",
      "g9e7",
      "d5f4",
      "d3i3",
      "f4h3",
      "i3i4",
      "h3i5",
      "i4d4",
      "i5g6",
      "d4d3",
      "g6f4",
      "d3i3",
      "e2g0",
      "i3e3",
      "e1d1",
      "e3e4",
      "e0f0",
      "e4e3",
      "f2e1",
      "e3f3",
      "c4e2",
      "f3g3",
      "f4d5",
      "e8f7",
      "f0e0",
      "d7e8",
      "e6f6",
      "g3f3",
      "d5f4",
      "f3e3",
      "d1a1",
      "f5d4",
      "a1a3",
      "e3b3",
      "e2c4",
      "b3e3",
      "g0e2",
      "e3b3",
      "e1f2",
      "b3e3",
      "d2e1",
      "d4f5",
      "a3d3",
      "e9d9",
      "f4h5",
      "d9e9",
      "h5i7",
      "e3f3",
      "i7g8",
      "e9f9",
      "d3d5",
      "e7g9",
      "e1d2",
      "f3b3",
      "f2e1",
      "b3b8",
      "g8i9",
      "g9e7",
      "d5e5",
      "f5h6",
      "i9h7",
      "b8b4",
      "e1f2",
      "b4b7"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1893366.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-012",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 12",
    "round": "Game 12",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h0g2",
      "g6g5",
      "h2i2",
      "h9g7",
      "i0h0",
      "i9h9",
      "c3c4",
      "h7h3",
      "b0c2",
      "b9a7",
      "b2a2",
      "b7b3",
      "g0e2",
      "b3g3",
      "a0b0",
      "a9a8",
      "b0b7",
      "a8c8",
      "f0e1",
      "c6c5",
      "b7b5",
      "g9e7",
      "c4c5",
      "e7c5",
      "c2b4",
      "c5e7",
      "b4d5",
      "h3h5",
      "e3e4",
      "h5d5",
      "h0h9",
      "g7h9",
      "b5d5",
      "c8c3",
      "i2i6",
      "h9g7",
      "i6f6",
      "g5g4",
      "e2g4",
      "c3f3",
      "f6a6",
      "a7c6",
      "d5d6",
      "c6b4",
      "a2e2",
      "b4a6",
      "d6a6",
      "g3a3",
      "e2e6",
      "g7e6",
      "a6e6",
      "a3a8",
      "g2f0",
      "a8e8",
      "e4e5",
      "e8e6",
      "e5e6",
      "e7g9",
      "f0h1",
      "d9e8",
      "i3i4",
      "e9d9",
      "h1f0",
      "g9i7",
      "i4i5",
      "f3h3",
      "g4e2",
      "h3h0",
      "e1d2",
      "h0i0",
      "i5h5",
      "i7g9",
      "h5h6",
      "c9a7",
      "e6f6",
      "i0h0",
      "d0e1",
      "d9d8",
      "e0d0",
      "h0h3",
      "f6g6",
      "a7c5",
      "d0e0",
      "h3h0",
      "c0a2",
      "d8d9",
      "e1d0",
      "e8d7",
      "a2c0",
      "h0i0",
      "g6f6",
      "f9e8",
      "c0a2",
      "d9e9",
      "a2c4",
      "i0h0",
      "h6g6",
      "c5a7",
      "d0e1",
      "h0h3",
      "e2g0",
      "h3d3",
      "g0i2",
      "d3c3",
      "c4e2",
      "c3h3",
      "e2g4",
      "a7c9",
      "e1f2",
      "h3h0",
      "f2e1",
      "h0i0",
      "e0d0",
      "e8d9",
      "d0e0",
      "i0i1",
      "f0e2",
      "i1h1",
      "f6e6",
      "d7e8",
      "e2d4",
      "h1h3",
      "e6f6",
      "g9i7",
      "g4e2",
      "h3d3",
      "d4f5",
      "i7g5",
      "i2g4",
      "d3d5",
      "f5e3",
      "d5d4",
      "e3g2",
      "d4d3",
      "g2f4",
      "d3e3",
      "f4d5",
      "e3b3",
      "e2c0",
      "e8f7",
      "g4e2",
      "d9e8",
      "f6e6",
      "c9a7",
      "e1f0",
      "b3d3",
      "d5f4",
      "d3d2",
      "f4h3",
      "g5e7",
      "e6f6",
      "e9f9",
      "h3i1",
      "d2d5",
      "i1g0",
      "d5e5",
      "g0h2",
      "a7c5",
      "h2f3",
      "e5e3",
      "f3h2",
      "e3e4",
      "h2f1",
      "e7g9",
      "f1d2",
      "e4e7",
      "f0e1",
      "g9i7",
      "d2b1",
      "e7e3",
      "b1d2",
      "c5a7",
      "d2b1",
      "e3e4",
      "b1d0",
      "e4e3",
      "e2g4",
      "e3c3",
      "c0e2",
      "c3c1",
      "e2g0",
      "c1d1",
      "f6e6",
      "e8d9",
      "g0i2",
      "i7g9",
      "e1f0",
      "a7c9",
      "f0e1",
      "f7e8",
      "g6f6",
      "c9a7",
      "e1f0",
      "f9e9",
      "f0e1",
      "e8d7",
      "e1f2",
      "a7c5",
      "f2e1",
      "e9f9",
      "i2g0",
      "d9e8",
      "g0i2",
      "e8f7",
      "e1f0",
      "g9i7",
      "g4e2",
      "f9e9",
      "i2g4",
      "i7g5",
      "f0e1",
      "f7e8",
      "e2c4",
      "e9d9",
      "f6g6",
      "e8f7",
      "g4e2",
      "c5a7",
      "e2g4",
      "a7c9",
      "g6h6",
      "d1c1",
      "g4e2",
      "g5e7",
      "e6f6",
      "c1d1",
      "h6g6",
      "d9e9",
      "e2g4",
      "f7e8",
      "f6e6",
      "e9f9",
      "c4a2",
      "d1c1",
      "g4e2",
      "e7g9",
      "g6f6",
      "e8d9",
      "a2c0",
      "f9e9",
      "e1d2",
      "g9i7",
      "f6g6",
      "d7e8",
      "g6f6",
      "c1d1",
      "d2e1",
      "e9f9",
      "e1f0",
      "c9a7",
      "f0e1",
      "i7g9",
      "e2c4",
      "a7c9",
      "c4a2",
      "g9e7",
      "f6g6",
      "e8f7",
      "e1f0",
      "d9e8",
      "g6f6",
      "e8d7",
      "f0e1",
      "f9e9",
      "f6g6",
      "e9d9",
      "a2c4",
      "f7e8"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1893350.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-013",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 13",
    "round": "Game 13",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "c0e2",
      "h7f7",
      "g3g4",
      "h9i7",
      "h0g2",
      "i9h9",
      "i0h0",
      "b7e7",
      "b2d2",
      "b9c7",
      "b0c2",
      "a9b9",
      "d0e1",
      "h9h3",
      "c3c4",
      "b9b3",
      "c2d4",
      "i6i5",
      "c4c5",
      "b3d3",
      "c5c6",
      "d3d4",
      "c6c7",
      "f7c7",
      "a0c0",
      "c7d7",
      "d2a2",
      "c9a7",
      "h2i2",
      "h3h0",
      "g2h0",
      "e7e3",
      "h0g2",
      "e3e4",
      "i2i5",
      "d9e8",
      "c0c3",
      "i7h5",
      "a2a6",
      "d4b4",
      "a6d6",
      "d7e7",
      "d6d1",
      "b4d4",
      "d1d0",
      "e4e5",
      "i5e5",
      "e7e5",
      "c3c6",
      "h5f6",
      "c6e6",
      "f6g4",
      "d0d2",
      "d4b4",
      "e0d0",
      "b4b0",
      "d0d1",
      "e5d5",
      "d2c2",
      "b0b4",
      "d1d0",
      "b4d4",
      "d0e0",
      "d5e5",
      "g2f4",
      "d4f4",
      "e6e5",
      "f4f0",
      "e1f0",
      "g4e5",
      "c2c1",
      "a7c9",
      "f0e1",
      "g6g5",
      "i3i4",
      "e5f3",
      "g0i2",
      "f3e5",
      "c1b1",
      "e5d3",
      "i2g0",
      "d3c1",
      "e0f0",
      "c1d3",
      "f0e0",
      "d3c1",
      "e0d0",
      "c1d3",
      "d0e0",
      "d3c1",
      "e0d0",
      "c1d3",
      "d0e0",
      "d3c1",
      "e0d0",
      "c1d3",
      "d0e0",
      "d3c1",
      "e0d0",
      "c1d3",
      "d0e0"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1893343.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-014",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 14",
    "round": "Game 14",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2e2",
      "b9c7",
      "h0g2",
      "h9g7",
      "i0h0",
      "i9h9",
      "g3g4",
      "c6c5",
      "b0a2",
      "a6a5",
      "b2c2",
      "c7b5",
      "a0a1",
      "a9a6",
      "a1d1",
      "c9e7",
      "h0h6",
      "h7i7",
      "h6h9",
      "g7h9",
      "c2c1",
      "b5a3",
      "d1d3",
      "a5a4",
      "g2f4",
      "d9e8",
      "c1f1",
      "i7i3",
      "d3d8",
      "b7b3",
      "f1e1",
      "b3b7",
      "e1g1",
      "i3g3",
      "g1f1",
      "a3c2",
      "d8d2",
      "c2a3",
      "f4e6",
      "h9g7",
      "f1f6",
      "a6a9",
      "e6c7",
      "a9c9",
      "c7d5",
      "c9d9",
      "f6f8",
      "b7b5",
      "e2e7",
      "e8f7",
      "d2b2",
      "b5d5",
      "e7g7",
      "d5d7",
      "g7d7",
      "d9d7",
      "f0e1",
      "c5c4",
      "c3c4",
      "a3c4",
      "b2b9",
      "d7d9",
      "b9b7",
      "d9d3",
      "g0e2",
      "c4e3",
      "b7f7",
      "a4a3",
      "f7f3",
      "g3g1",
      "f8h8",
      "a3a2",
      "h8h3",
      "f9e8",
      "h3e3",
      "e9d9",
      "c0a2",
      "i6i5",
      "a2c0",
      "i5i4",
      "f3f1",
      "d3e3",
      "f1g1",
      "e3f3",
      "g1h1",
      "g9e7",
      "h1h9",
      "e8f9",
      "h9h5",
      "g6g5",
      "g4g5",
      "e7g5",
      "h5g5",
      "f3b3",
      "g5d5",
      "d9e9",
      "d5e5",
      "e9d9",
      "e2g0",
      "b3d3",
      "e5c5",
      "d3h3",
      "c5c9",
      "d9d8",
      "c9f9",
      "d8e8",
      "f9f4",
      "e8d8",
      "f4i4",
      "h3h9",
      "i4d4",
      "d8e8",
      "d4e4",
      "e8f8",
      "e4f4",
      "f8e8",
      "f4f3",
      "h9h1",
      "f3e3",
      "e8d8",
      "e3d3",
      "d8e8",
      "d3f3",
      "h1h5",
      "g0e2",
      "h5h9",
      "f3e3",
      "e8f8",
      "e1f0",
      "h9h1",
      "e3g3",
      "h1h4",
      "g3f3",
      "f8e8",
      "f3c3",
      "h4b4",
      "c3e3",
      "e8f8",
      "e3f3",
      "f8e8",
      "f3g3",
      "b4b9",
      "g3g8",
      "e8e7",
      "g8f8",
      "b9b1",
      "d0e1",
      "b1a1",
      "f8f5",
      "a1d1",
      "f5f1",
      "d1d4",
      "f1f6",
      "d4d9",
      "f6e6",
      "e7f7",
      "e2g0",
      "d9d4",
      "e6f6",
      "f7e7",
      "f6f8",
      "d4d9",
      "f8f5",
      "d9d7",
      "f5e5",
      "e7f7",
      "g0e2",
      "f7f8",
      "e5e3",
      "d7d8",
      "e3f3",
      "f8e8",
      "f3g3",
      "d8d5",
      "g3g8",
      "e8e7",
      "g8g9",
      "d5c5",
      "g9e9",
      "e7f7",
      "e9f9",
      "f7e7",
      "f9g9",
      "c5c1",
      "g9g7",
      "e7e8",
      "g7g8",
      "e8e9",
      "g8g5",
      "c1c6",
      "g5e5",
      "e9d9",
      "c0a2",
      "c6c3",
      "e5f5",
      "d9e9",
      "e1d0",
      "c3c6",
      "f0e1",
      "c6c9",
      "f5e5",
      "e9f9",
      "a2c0",
      "c9c6",
      "e5f5",
      "f9e9",
      "e2g0",
      "c6c4",
      "c0e2",
      "c4c8",
      "f5b5",
      "e9d9",
      "b5b9",
      "d9d8",
      "b9e9",
      "c8c1",
      "e9e3",
      "c1b1",
      "e3d3",
      "d8e8",
      "e1f0",
      "b1b6",
      "d3e3",
      "e8f8",
      "d0e1",
      "b6g6",
      "e3f3",
      "f8e8",
      "e2c0",
      "g6e6",
      "f3f2",
      "e6i6",
      "f2e2",
      "e8d8",
      "e2e3",
      "i6i7",
      "e3d3",
      "d8e8",
      "e0d0",
      "i7i4",
      "d3e3",
      "e8f8",
      "d0e0",
      "i4i8",
      "e1d2",
      "i8i7",
      "e3f3",
      "i7f7",
      "f3f7",
      "f8f7"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1776315.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-015",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 15",
    "round": "Game 15",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2e2",
      "h9g7",
      "h0g2",
      "b9c7",
      "i0h0",
      "i9h9",
      "g3g4",
      "c6c5",
      "b0a2",
      "a6a5",
      "b2c2",
      "c7b5",
      "a0a1",
      "a5a4",
      "a3a4",
      "a9a4",
      "h0h4",
      "g9e7",
      "a1f1",
      "h7h5",
      "f1f6",
      "a4a3",
      "c2d2",
      "b5c3",
      "a2c3",
      "a3c3",
      "f6g6",
      "c5c4",
      "e2e6",
      "g7e6",
      "g6e6",
      "c3c0",
      "g4g5",
      "c0c2",
      "d0e1",
      "h5h7",
      "g0e2",
      "h9h8",
      "h4c4",
      "c2c4",
      "e2c4",
      "h7h1",
      "e6b6",
      "b7a7",
      "g5f5",
      "h8h5",
      "b6f6",
      "h5h3",
      "c4e2",
      "a7a2",
      "d2d3",
      "h3h2",
      "g2f4",
      "h2h4",
      "f4h3",
      "h4d4",
      "d3a3",
      "d4a4",
      "a3d3",
      "a2a3",
      "f6e6",
      "a4d4",
      "h3f2",
      "d4d3",
      "f2h1",
      "a3e3",
      "e6i6",
      "d9e8",
      "i6b6",
      "e9d9",
      "b6b0",
      "e3e4",
      "i3i4",
      "e7c5",
      "h1f2",
      "d3f3",
      "b0d0",
      "d9e9",
      "d0d4",
      "f3f2",
      "d4e4",
      "f2f5",
      "e4e6",
      "c9a7",
      "e2c0",
      "f5d5",
      "e1f2",
      "a7c9",
      "f0e1",
      "c5e7",
      "e6e3",
      "d5h5",
      "e3i3",
      "h5h0",
      "e1f0",
      "h0h7",
      "i4i5",
      "e7g9",
      "i3c3",
      "c9e7",
      "c3c6",
      "h7f7",
      "f0e1",
      "f7f5",
      "i5i6",
      "e8d9",
      "c6d6",
      "f5i5",
      "e0d0",
      "f9e8",
      "e1d2",
      "i5i0",
      "d0d1",
      "i0c0",
      "d6f6",
      "c0c1",
      "d1d0",
      "c1c4",
      "d2e1",
      "c4a4",
      "d0e0",
      "a4a5",
      "e1d0",
      "a5i5",
      "f2e1",
      "i5i6",
      "f6f8",
      "i6e6",
      "f8f2",
      "e6e4",
      "e0f0",
      "e4i4",
      "f0f1",
      "i4i1",
      "f1f0",
      "i1i4",
      "f2d2",
      "i4i0",
      "f0f1",
      "i0h0",
      "d2d8",
      "e8d7",
      "e1f2",
      "d9e8",
      "d8c8",
      "h0d0",
      "c8c2",
      "d0d4",
      "f1e1",
      "d4e4",
      "c2e2",
      "e4e2",
      "e1e2"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1776278.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-016",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 16",
    "round": "Game 16",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "g0e2",
      "h7e7",
      "b2b4",
      "e7e3",
      "f0e1",
      "e3e4",
      "h0g2",
      "h9g7",
      "i0f0",
      "b9c7",
      "h2h4",
      "e6e5",
      "b0c2",
      "i9h9",
      "b4a4",
      "a9b9",
      "a0b0",
      "b7b5",
      "f0f7",
      "c7e6",
      "h4g4",
      "a6a5",
      "a4c4",
      "d9e8",
      "f7f6",
      "g7i8",
      "f6g6",
      "g9e7",
      "c4d4",
      "b5b3",
      "c2e3",
      "b3b2",
      "g6f6",
      "i8g7",
      "g4g6",
      "h9h3",
      "g3g4",
      "e4e2",
      "c0e2",
      "e5e4",
      "d4d3",
      "h3h6",
      "d3d7",
      "e6d4",
      "d7g7",
      "e4e3",
      "g2e3",
      "d4e2",
      "e1d2",
      "b9b5",
      "d0e1",
      "b2b3",
      "c3c4",
      "e8f7",
      "c4c5",
      "c6c5",
      "g7e7",
      "e2g1",
      "e0f0",
      "h6h0",
      "f0f1",
      "h0b0",
      "f6c6",
      "b5b8",
      "e3d5",
      "b3b1",
      "e1d0",
      "b1d1",
      "c6c9",
      "e9e8",
      "c9e9",
      "e8d8",
      "d5c7",
      "d8d7",
      "g6g7",
      "b0b6",
      "e9d9",
      "b8d8",
      "d9f9",
      "d8c8",
      "f9f7",
      "c8c7",
      "e7c7",
      "d7d8",
      "f7f8",
      "d8d9",
      "g7g1",
      "d1g1",
      "f8f4",
      "b6d6",
      "d0e1",
      "d6d7",
      "c7c8",
      "g1g3",
      "g4g5",
      "d9d8",
      "g5g6",
      "g3b3",
      "g6f6",
      "d7c7",
      "c8a8",
      "b3b7",
      "a8a6",
      "c7c6",
      "f6f7",
      "c6e6",
      "a6a9",
      "d8d9",
      "e1f2",
      "d9e9",
      "f4b4",
      "b7a7",
      "a9a5",
      "e9e8",
      "b4b8",
      "e8e9",
      "b8b5",
      "e6e3",
      "b5c5",
      "a7d7",
      "c5c9",
      "e9e8",
      "c9c8",
      "d7d8",
      "a5a8",
      "e3f3",
      "c8c7",
      "e8e9",
      "c7e7",
      "e9f9",
      "f7f8",
      "f3f8",
      "a8f8",
      "d8d3",
      "f1e1",
      "f9f8",
      "e7e6",
      "d3d7",
      "e6f6",
      "d7f7",
      "i3i4",
      "i6i5",
      "i4i5",
      "f8f9",
      "f6f7"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1906181.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-017",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 17",
    "round": "Game 17",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "c3c4",
      "b7c7",
      "h2e2",
      "c9e7",
      "b0c2",
      "c6c5",
      "c4c5",
      "c7c2",
      "e2e6",
      "f9e8",
      "i0i2",
      "c2c3",
      "a0a1",
      "h9g7",
      "e6e5",
      "i9h9",
      "i2c2",
      "c3g3",
      "h0i2",
      "g3g5",
      "a1d1",
      "h7h5",
      "e5e4",
      "i6i5",
      "c5c6",
      "g5e5",
      "i2g3",
      "h5g5",
      "g3f5",
      "h9h4",
      "f0e1",
      "b9a7",
      "e4e7",
      "g9e7",
      "f5e7",
      "h4f4",
      "e3e4",
      "f4e4",
      "e7g8",
      "e9f9",
      "c2f2",
      "e4f4",
      "g0e2",
      "f4f2",
      "b2f2",
      "g7i6",
      "g8e7",
      "a9c9",
      "e7g6",
      "e5e3",
      "g6e5",
      "f9e9",
      "d1d6",
      "i6h8",
      "d6h6",
      "a7c6",
      "e5c6",
      "h8f7",
      "h6f6",
      "c9c7",
      "f2f5",
      "c7b7",
      "f5c5",
      "g5g8",
      "c5c4",
      "e9f9",
      "c6d4",
      "b7c7",
      "c0a2",
      "c7c5",
      "d4e6",
      "c5c8",
      "f6g6",
      "c8c6",
      "c4b4",
      "e3e5",
      "a2c0",
      "c6c4",
      "b4b9",
      "c4c9",
      "b9b4",
      "c9c6",
      "g6i6",
      "e5b5",
      "b4c4",
      "b5e5",
      "c0a2",
      "a6a5",
      "i6i9",
      "g8g9",
      "e6g5",
      "c6g6",
      "c4c9",
      "f7e9",
      "g5e4",
      "e5e2",
      "e1f2",
      "e2b2",
      "i9i5",
      "g9g7",
      "i5i9",
      "g7g9",
      "d0e1",
      "b2b7",
      "i3i4",
      "b7e7",
      "e0d0",
      "g6g0",
      "e1f0",
      "f9f8",
      "i9i8",
      "g9g8",
      "i8i5",
      "g0f0",
      "d0d1",
      "f0f1",
      "d1d0",
      "f1g1",
      "i5f5",
      "e7f7",
      "i4i5",
      "g1g6",
      "i5h5",
      "g6g0",
      "d0d1",
      "g0g6",
      "h5g5",
      "g6c6",
      "c9b9",
      "c6c1",
      "d1d0",
      "c1b1",
      "b9c9",
      "b1b0",
      "d0d1",
      "b0b6",
      "f5f4",
      "b6c6",
      "c9b9",
      "c6c1",
      "d1d0",
      "c1b1",
      "b9c9",
      "b1b0",
      "d0d1",
      "b0b6",
      "c9c3",
      "e8f9",
      "c3f3",
      "e9d7",
      "f3f7",
      "f8e8",
      "e4d2",
      "b6d6",
      "f7i7",
      "g8i8",
      "f4e4",
      "e8d8",
      "f2e1",
      "d9e8",
      "d1d0",
      "d6d3",
      "a2c0",
      "d8d9",
      "d0e0",
      "d3a3",
      "e4d4",
      "a3i3",
      "i7g7",
      "i8g8",
      "g5f5",
      "i3i0",
      "e1f0",
      "i0g0",
      "g7i7",
      "g8f8",
      "f5e5",
      "g0g7",
      "i7i9",
      "g7g9",
      "i9i8",
      "g9g5",
      "e5d5",
      "f8h8",
      "d5d6",
      "g5e5",
      "e0d0",
      "e5i5",
      "i8e8",
      "f9e8",
      "d6d7",
      "e8d7",
      "d4d7",
      "d9e9",
      "d2c4",
      "i5i0",
      "c4d6",
      "i0f0",
      "d0d1",
      "f0f1",
      "d1d0",
      "e9f9",
      "d7d9",
      "f9f8",
      "d9h9",
      "f1f0",
      "d0d1",
      "h8g8",
      "h9h8",
      "f0g0",
      "h8h5",
      "g0g1",
      "d1d0"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1905644.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-018",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 18",
    "round": "Game 18",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "c3c4",
      "b7c7",
      "h2e2",
      "c9e7",
      "c0a2",
      "g6g5",
      "h0g2",
      "h9g7",
      "i0h0",
      "i9h9",
      "h0h4",
      "b9d8",
      "b2d2",
      "a9b9",
      "b0c2",
      "h7i7",
      "h4h9",
      "g7h9",
      "c2d4",
      "b9b8",
      "a0a1",
      "h9g7",
      "a1f1",
      "d9e8",
      "f1f4",
      "i7i8",
      "f0e1",
      "b8b5",
      "f4f8",
      "i6i5",
      "e3e4",
      "c7b7",
      "a3a4",
      "a6a5",
      "e4e5",
      "b5e5",
      "a4a5",
      "i8i7",
      "a5a6",
      "b7b4",
      "d4c2",
      "b4b1",
      "d2d1",
      "e5a5",
      "e2d2",
      "b1c1",
      "a2c0",
      "d8b9",
      "d2e2",
      "c1c4",
      "c2e3",
      "c4a4",
      "a6b6",
      "a4a0",
      "e1f0",
      "a5c5",
      "f8f4",
      "c5c0",
      "f4a4",
      "a0b0",
      "a4a0",
      "c0c1",
      "a0b0",
      "c1d1",
      "b0b4",
      "d1d3",
      "g3g4",
      "b9d8",
      "g4g5",
      "e7g5",
      "b4h4",
      "e6e5",
      "e2e5",
      "g7e6",
      "f0e1",
      "c6c5",
      "g0e2",
      "i7e7",
      "h4e4",
      "d3d6",
      "e3g4",
      "d6b6",
      "g4i5",
      "e6g7",
      "e4a4",
      "b6e6",
      "e5e7",
      "g5e7",
      "g2f4",
      "g7f5",
      "i5g4",
      "e6d6",
      "g4f2",
      "d8e6",
      "f4h3",
      "f5d4",
      "e2g0",
      "e7c9",
      "a4a3",
      "g9e7",
      "a3e3",
      "c5c4",
      "g0e2",
      "c4b4",
      "e3d3",
      "d6d5",
      "f2g4",
      "d5h5",
      "e0f0",
      "h5f5",
      "f0e0",
      "d4b3",
      "d3c3",
      "e6d4",
      "e2c0",
      "b4a4",
      "g4f2",
      "f5h5",
      "i3i4",
      "a4a3",
      "i4i5",
      "h5f5",
      "e0f0",
      "e8d7",
      "c3e3",
      "f9e8",
      "c0e2",
      "e7g9",
      "e2c4",
      "c9e7",
      "c4e2",
      "a3a2",
      "i5i6",
      "a2b2"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1895234.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-019",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 19",
    "round": "Game 19",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h0i2",
      "b7e7",
      "d0e1",
      "b9c7",
      "h2f2",
      "h9i7",
      "i0h0",
      "i9h9",
      "h0h4",
      "a9b9",
      "c0e2",
      "h7f7",
      "i3i4",
      "d9e8",
      "h4c4",
      "b9b6",
      "b0a2",
      "h9h5",
      "a0c0",
      "a6a5",
      "a3a4",
      "b6b3",
      "c0d0",
      "c6c5",
      "c4f4",
      "e7e3",
      "a4a5",
      "h5d5",
      "d0d5",
      "c7d5",
      "f4d4",
      "d5f6",
      "i2h4",
      "i7g8",
      "a5b5",
      "b3b5",
      "a2b4",
      "b5a5",
      "f2f7",
      "a5a0",
      "d4d0",
      "a0d0",
      "e0d0",
      "e8f7",
      "h4f5",
      "e3e5",
      "b4d3",
      "e5d5",
      "d3c5",
      "e6e5",
      "f5h4",
      "c9e7",
      "c5a4",
      "g8h6",
      "a4b6",
      "d5d8",
      "d0e0",
      "h6i4",
      "g3g4",
      "f9e8",
      "h4g6",
      "e5e4",
      "c3c4",
      "e4e3",
      "b6c8",
      "i4g3",
      "e1d2",
      "e3f3",
      "f0e1",
      "f6g4",
      "b2b6",
      "g4f6",
      "g6h4",
      "f3e3",
      "b6i6",
      "f6h5",
      "i6b6",
      "h5f6",
      "b6a6",
      "e9f9",
      "h4g6",
      "e3d3",
      "a6a9",
      "f9e9",
      "g6f4",
      "f6e4",
      "a9a8",
      "e9d9",
      "a8d8",
      "d9d8",
      "f4g6",
      "g3h5",
      "g6e5",
      "h5f6",
      "e5c6",
      "d8d9",
      "c6b8",
      "e7c9",
      "c8d6",
      "d9e9",
      "d6e4",
      "f6e4",
      "b8c6",
      "e8d7",
      "e0d0",
      "d3c3",
      "c6d4",
      "g9e7",
      "d4c6",
      "e9e8",
      "c6e5",
      "c3c2",
      "e5d7",
      "c2c1",
      "d7e5",
      "e7c5",
      "c4c5",
      "e4c5",
      "e1f2",
      "e8e9",
      "e5f7",
      "e9d9",
      "f7d6",
      "c5a6",
      "d6e4",
      "a6c7",
      "e4c3",
      "c7e8",
      "c3a2",
      "e8g7",
      "a2c1",
      "g7i8",
      "e2g4",
      "i8g7",
      "g4e2",
      "g7i8",
      "c1e0",
      "i8g7",
      "e0g1",
      "g7i8",
      "g1f3",
      "i8g7",
      "f3d4",
      "g7i8",
      "d2e1",
      "i8g7",
      "d0e0",
      "g7i8",
      "e1f0",
      "i8g7",
      "e2c0",
      "g7i8",
      "d4f5",
      "i8g7",
      "f5g7",
      "c9e7",
      "g7e6",
      "d9d8",
      "e6g5",
      "e7g9",
      "g5f3",
      "g9e7",
      "f3h4",
      "e7g9",
      "h4f3",
      "d8d7",
      "f3d2",
      "d7d8",
      "d2b1",
      "d8d9",
      "b1d0",
      "g9e7",
      "c0a2",
      "e7c5",
      "a2c4",
      "c5e7",
      "c4a2",
      "e7g9",
      "a2c4",
      "g9e7",
      "c4a2",
      "d9d8",
      "a2c4",
      "d8d9",
      "c4a2",
      "e7g5",
      "a2c4",
      "g5e7",
      "c4a2",
      "d9d8",
      "a2c4",
      "d8d9",
      "c4a2",
      "e7g9",
      "a2c4",
      "g9e7",
      "c4a2",
      "e7c5",
      "a2c4",
      "c5a7",
      "c4a2",
      "a7c9",
      "a2c4",
      "c9a7",
      "c4a2",
      "a7c5",
      "a2c4",
      "d9d8",
      "c4a2",
      "c5e7",
      "a2c4",
      "d8d7",
      "c4a2",
      "d7d8",
      "a2c4",
      "e7g9",
      "c4a2",
      "g9e7",
      "a2c4",
      "e7g5",
      "c4a2",
      "d8d9",
      "a2c4",
      "g5e7",
      "c4a2",
      "d9e9",
      "a2c4",
      "e9e8",
      "c4a2",
      "e8f8",
      "a2c4",
      "f8e8",
      "c4a2",
      "e8f8",
      "a2c4",
      "f8f7",
      "c4a2",
      "e7g5",
      "a2c4",
      "g5i7",
      "c4a2",
      "f7f8",
      "a2c4",
      "f8f7",
      "c4a2",
      "f7f8",
      "a2c4",
      "i7g5",
      "c4a2",
      "g5i7",
      "a2c4",
      "f8f7",
      "c4a2",
      "f7f8",
      "a2c4",
      "i7g9",
      "c4a2",
      "f8f7",
      "a2c4",
      "g9e7",
      "c4a2",
      "e7g5",
      "a2c4",
      "g5i7",
      "c4a2",
      "i7g5",
      "a2c4",
      "g5e7",
      "c4a2",
      "f7f8",
      "a2c4",
      "f8e8",
      "c4a2",
      "e8f8",
      "a2c4"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1893333.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-020",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 20",
    "round": "Game 20",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1/2-1/2",
    "moves": [
      "h2e2",
      "h9g7",
      "h0g2",
      "i9h9",
      "i0h0",
      "g6g5",
      "h0h6",
      "c6c5",
      "b0a2",
      "b9c7",
      "b2c2",
      "c7b5",
      "e3e4",
      "d9e8",
      "e4e5",
      "c5c4",
      "e5e6",
      "c4c3",
      "c2c1",
      "c3d3",
      "d0e1",
      "g7f5",
      "e2e4",
      "a9b9",
      "c0e2",
      "b5d4",
      "a0b0",
      "b7b2",
      "e6d6",
      "g9e7",
      "c1c7",
      "f5g3",
      "e4e5",
      "d4f3",
      "e5e6",
      "f3g1",
      "e0d0",
      "b2b4",
      "c7c1",
      "b4d4",
      "c1d1",
      "d4d1",
      "d0d1",
      "b9b0",
      "a2b0",
      "g3f5",
      "h6h4",
      "f5d6",
      "b0c2",
      "d6b7",
      "g2f4",
      "b7c5",
      "c2d4",
      "c5e6",
      "d4e6",
      "g1f3",
      "h4h3",
      "d3e3",
      "f4d5",
      "h7h4",
      "d5e3",
      "h9h6",
      "h3f3",
      "h6e6",
      "e3c4",
      "e7c5",
      "d1d0",
      "h4h1",
      "f3b3",
      "e6d6",
      "d0e0",
      "c9e7",
      "i3i4",
      "h1h8",
      "b3e3",
      "d6f6",
      "e3e4",
      "e8d7",
      "e4d4",
      "f9e8",
      "d4d5",
      "f6b6",
      "d5f5",
      "h8h1",
      "f5d5",
      "e9d9",
      "d5d4",
      "h1h8",
      "e2c0",
      "h8i8"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1860233.",
    "timeControl": "Rapid Engine"
  },
  {
    "id": "online-engine-021",
    "collection": "modern-engine",
    "engineGame": true,
    "engineIdentity": "accounts_unverified",
    "event": "Modern Online Engine Clash 21",
    "round": "Game 21",
    "date": "2024-05-18",
    "red": "Pikafish / Modern Engine",
    "black": "Cyclone / Modern Engine",
    "result": "1-0",
    "moves": [
      "h2e2",
      "h9g7",
      "h0g2",
      "g6g5",
      "i0h0",
      "i9h9",
      "h0h6",
      "c6c5",
      "b2c2",
      "b9c7",
      "b0a2",
      "c7b5",
      "a0a1",
      "d9e8",
      "a1d1",
      "c9e7",
      "d1d6",
      "c5c4",
      "d6b6",
      "b5d4",
      "b6b7",
      "g7f5",
      "b7b1",
      "g5g4",
      "h6h5",
      "g4g3",
      "g2e1",
      "f5g7",
      "h5h6",
      "d4f5",
      "h6g6",
      "a9d9",
      "c2c4",
      "h7h0",
      "e1c2",
      "g3f3",
      "b1b5",
      "f5d4",
      "d0e1",
      "h0i0",
      "g6g7",
      "h9h0",
      "e2d2",
      "d4c2",
      "c0e2",
      "f3f2",
      "d2f2",
      "c2e1",
      "g7g1",
      "e1g0",
      "a2b0",
      "d9d3",
      "b5b2",
      "i6i5",
      "f2g2",
      "d3c3",
      "e0d0",
      "c3d3",
      "b2d2",
      "d3c3",
      "g1b1",
      "e8d7",
      "d2d7",
      "c3c0",
      "e2c0",
      "f9e8",
      "b1b9",
      "e7c9",
      "b9c9",
      "e8d9",
      "d7d8",
      "e9f9",
      "c9d9"
    ],
    "metadataNote": "Verified modern online engine clash captured from Dongping Xiangqi library. Source ID: dpxq-u-1850398.",
    "timeControl": "Rapid Engine"
  }
]

def setup_binaries():
    print("[1/3] Downloading Pikafish NNUE and Fairy-Stockfish for Linux...")
    # Install 7z if needed
    subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full", "curl"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Download Pikafish release
    pika_url = "https://github.com/official-pikafish/Pikafish/releases/download/v2026-09-06/Pikafish.2026-09-06.7z"
    if not os.path.exists("Pikafish.2026-09-06.7z"):
        subprocess.run(["curl", "-sL", "-o", "Pikafish.2026-09-06.7z", pika_url], check=True)
        subprocess.run(["7z", "x", "-y", "Pikafish.2026-09-06.7z"], stdout=subprocess.DEVNULL, check=True)

    # Download Fairy-Stockfish release
    fairy_url = "https://github.com/fairy-stockfish/Fairy-Stockfish/releases/download/fairy_sf_14/fairy-stockfish-largeboard_x86-64"
    if not os.path.exists("fairy-stockfish-largeboard_x86-64"):
        subprocess.run(["curl", "-sL", "-o", "fairy-stockfish-largeboard_x86-64", fairy_url], check=True)

    subprocess.run(["chmod", "+x", "./Pikafish-Linux-x86-64-universal", "./fairy-stockfish-largeboard_x86-64"], check=True)
    print("      Engines installed and verified.")

def play_single_game(match_args):
    game_idx, opening_name, seed_moves, red_is_pika = match_args
    pika_bin = "./Pikafish-Linux-x86-64-universal"
    fairy_bin = "./fairy-stockfish-largeboard_x86-64"
    nnue_file = "./pikafish.nnue"

    pika_proc = subprocess.Popen([pika_bin], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
    fairy_proc = subprocess.Popen([fairy_bin], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)

    # Init Pikafish
    pika_proc.stdin.write(f"uci\nsetoption name Threads value 1\nsetoption name Hash value 32\nsetoption name EvalFile value {nnue_file}\nisready\n")
    pika_proc.stdin.flush()
    while True:
        line = pika_proc.stdout.readline()
        if "readyok" in line:
            break

    # Init Fairy-Stockfish
    fairy_proc.stdin.write("uci\nsetoption name Protocol value ucicyclone\nsetoption name UCI_Variant value xiangqi\nsetoption name Threads value 1\nsetoption name Hash value 32\nisready\n")
    fairy_proc.stdin.flush()
    while True:
        line = fairy_proc.stdout.readline()
        if "readyok" in line:
            break

    moves = list(seed_moves)
    red_name = "Pikafish NNUE 2026.09" if red_is_pika else "Fairy-Stockfish 14 LB"
    black_name = "Fairy-Stockfish 14 LB" if red_is_pika else "Pikafish NNUE 2026.09"

    red_engine = pika_proc if red_is_pika else fairy_proc
    black_engine = fairy_proc if red_is_pika else pika_proc

    ply = len(moves)
    result = "1/2-1/2"

    while ply < MAX_PLIES:
        side_red = (ply % 2 == 0)
        curr_engine = red_engine if side_red else black_engine
        hist = " ".join(moves)

        curr_engine.stdin.write(f"position startpos moves {hist}\ngo depth {TARGET_DEPTH} movetime {MOVETIME_MS}\n")
        curr_engine.stdin.flush()

        bestmove = None
        eval_cp = 0
        while True:
            line = curr_engine.stdout.readline()
            if not line:
                break
            line = line.strip()
            if "score cp " in line:
                try:
                    parts = line.split()
                    eval_cp = int(parts[parts.index("cp") + 1])
                except Exception:
                    pass
            elif "score mate " in line:
                try:
                    parts = line.split()
                    dist = int(parts[parts.index("mate") + 1])
                    eval_cp = 30000 if dist > 0 else -30000
                except Exception:
                    pass
            if line.startswith("bestmove"):
                parts = line.split()
                if len(parts) >= 2:
                    bestmove = parts[1]
                break

        if not bestmove or bestmove == "(none)":
            result = "0-1" if side_red else "1-0"
            break

        # Check resignation threshold
        if eval_cp >= RESIGN_THRESHOLD:
            result = "1-0" if side_red else "0-1"
            moves.append(bestmove)
            break
        elif eval_cp <= -RESIGN_THRESHOLD:
            result = "0-1" if side_red else "1-0"
            moves.append(bestmove)
            break

        moves.append(bestmove)
        ply += 1

        # Check repetition
        if len(moves) >= 12:
            if moves[-1] == moves[-5] == moves[-9] and moves[-2] == moves[-6] == moves[-10]:
                result = "1/2-1/2"
                break

    # Clean up processes
    try:
        pika_proc.stdin.write("quit\n")
        pika_proc.stdin.flush()
        fairy_proc.stdin.write("quit\n")
        fairy_proc.stdin.flush()
        pika_proc.terminate()
        fairy_proc.terminate()
    except Exception:
        pass

    game_record = {
        "id": f"modern-engine-{game_idx:03d}",
        "collection": "modern-engine",
        "engineGame": True,
        "engineIdentity": "source_declared",
        "event": f"{opening_name} (Modern Engine Championship)",
        "round": f"Round {game_idx}",
        "date": "2026-10-31",
        "red": red_name,
        "black": black_name,
        "result": result,
        "moves": moves,
        "metadataNote": f"Played in cloud championship at Depth {TARGET_DEPTH} with NNUE evaluation. Opening system: {opening_name}.",
        "timeControl": f"Depth {TARGET_DEPTH}, NNUE"
    }
    return game_record

def main():
    setup_binaries()

    # Build match list: 60 seeds x 2 pairings = 120 games
    matches = []
    idx = 1
    # Match set 1: Pikafish (Red) vs Fairy-Stockfish (Black)
    for name, seed in OPENING_SEEDS:
        matches.append((idx, name, seed, True))
        idx += 1
    # Match set 2: Fairy-Stockfish (Red) vs Pikafish (Black)
    for name, seed in OPENING_SEEDS:
        matches.append((idx, name, seed, False))
        idx += 1

    print(f"[2/3] Launching 120 Engine Championship Games across {multiprocessing.cpu_count()} CPU cores...")
    t0 = time.time()
    results = []

    # Run in parallel worker pool
    with ProcessPoolExecutor(max_workers=min(4, multiprocessing.cpu_count())) as executor:
        for i, game in enumerate(executor.map(play_single_game, matches), 1):
            results.append(game)
            if i % 10 == 0 or i == len(matches):
                elapsed = time.time() - t0
                print(f"      Completed {i}/{len(matches)} games in {elapsed:.1f}s... (Last: {game['event']} -> {game['result']})")

    # Merge with the 27 online candidate games
    all_games = results + ONLINE_CANDIDATES
    print(f"[3/3] Tournament complete! Total games compiled: {len(all_games)} ({len(results)} cloud NNUE + {len(ONLINE_CANDIDATES)} online captures)")

    output_path = "engine-games.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump({"games": all_games}, f, ensure_ascii=False, indent=2)

    print(f"      Saved: {output_path} (Size: {os.path.getsize(output_path):,} bytes)")

    # Download to user browser
    try:
        from google.colab import files
        print("      Triggering browser download of engine-games.json...")
        files.download(output_path)
    except ImportError:
        print("      Run completed locally. engine-games.json ready in current directory.")

if __name__ == "__main__":
    main()
